# 22 — Time-Domain Analysis

Visualise the raw vibration waveform at different lifecycle stages and inspect amplitude distributions and run-to-run variability.

**Dataset**: XJTU-SY — Bearing 1_1 (horizontal accelerometer)  
**API**: `assay.plot_timeseries()` · `assay.plot_distribution()` · `assay.plot_variability()`

In [ ]:
import warnings, logging, sys
from pathlib import Path

warnings.filterwarnings("ignore")
logging.getLogger("isa_phm").setLevel(logging.ERROR)

# Add python-wrapper package to path (works from repo root or notebook folder).
def _ensure_local_package() -> None:
    cwd = Path.cwd().resolve()
    for root in [cwd, *cwd.parents]:
        if (root / "isa_phm").is_dir() and (root / "pyproject.toml").exists():
            root_s = str(root)
            if root_s not in sys.path:
                sys.path.insert(0, root_s)
            return
    raise RuntimeError("Could not locate python-wrapper root with isa_phm package.")

_ensure_local_package()

from isa_phm import ISAWrapper
from bokeh.io import output_notebook
from bokeh.plotting import show as bokeh_show
output_notebook()

ISA_JSON = Path(r"G:/ISA/Datasets/XJTU-SY_Bearing_Datasets/XJTU-SY_Bearing_Datasets/XJTU-SY Bearing Datasets-ISA-PHM-Out.json")
DATA_ROOT = ISA_JSON.parent
print("ISA-JSON:", ISA_JSON)
print("DATA_ROOT:", DATA_ROOT)
print("Exists  :", ISA_JSON.exists())


In [ ]:
wrapper = ISAWrapper(path=ISA_JSON, data_root=DATA_ROOT, strict_validation=False, cache_maxsize=10)

study = wrapper.study("Bearing 1_1")
assay = study.assay(1)   # horizontal accelerometer

all_runs  = assay.list_runs()
first_run = all_runs[0]
last_run  = all_runs[-1]
mid_run   = all_runs[len(all_runs) // 2]

print(f"Assay : {assay.assay_id}  ({assay.run_count} runs)")
print(f"First : {first_run.run_id}  (#{first_run.run_number})")
print(f"Mid   : {mid_run.run_id}   (#{mid_run.run_number})")
print(f"Last  : {last_run.run_id}  (#{last_run.run_number})")


## 1. Raw waveform — healthy baseline (run 1)

The healthy waveform is near-Gaussian with low amplitude.  
Pan and zoom with the bokeh toolbar to inspect individual cycles.

In [ ]:
fig = assay.plot_timeseries(
    run_id=first_run.run_id,
    file_type="raw",
    title=f"Time-series — Healthy (run {first_run.run_number})",
)
bokeh_show(fig)

## 2. Raw waveform — mid-life

In [ ]:
fig = assay.plot_timeseries(
    run_id=mid_run.run_id,
    file_type="raw",
    title=f"Time-series — Mid-life (run {mid_run.run_number})",
)
bokeh_show(fig)

## 3. Raw waveform — near failure

The degraded signal shows strong periodic impact trains — fault-induced impulses at the ball-pass frequency.

In [ ]:
fig = assay.plot_timeseries(
    run_id=last_run.run_id,
    file_type="raw",
    title=f"Time-series — Near Failure (run {last_run.run_number})",
)
bokeh_show(fig)

## 4. Amplitude distribution — healthy vs degraded

A healthy bearing has a narrow, symmetric distribution.  
At failure the distribution widens and develops heavy tails (high kurtosis).

In [ ]:
fig = assay.plot_distribution(
    run_id=first_run.run_id,
    file_type="raw",
    title=f"Amplitude Distribution — Healthy (run {first_run.run_number})",
)
bokeh_show(fig)

In [ ]:
fig = assay.plot_distribution(
    run_id=last_run.run_id,
    file_type="raw",
    title=f"Amplitude Distribution — Near Failure (run {last_run.run_number})",
)
bokeh_show(fig)

## 5. Run-to-run variability

`plot_variability()` shows the amplitude spread across all runs as a boxplot.  
The growing interquartile range near the end reflects increasing fault severity.

In [ ]:
fig = assay.plot_variability(
    run_ids=[r.run_id for r in all_runs],
    file_type="raw",
)
bokeh_show(fig)

## 6. Run-level navigation (`RunProxy`)

Use `assay.run(run_id)` when you want one run as a first-class object.


In [ ]:
run_proxy = assay.run(first_run.run_id)
print(f"RunProxy -> run_id={run_proxy.run_id}, run_number={run_proxy.run_number}")
run_df = run_proxy.load_dataframe(file_type="raw")
print(run_df.head(5))


## 7. Missing-value visualization

Quick heatmap-style check of missing sample patterns within one run.


In [ ]:
fig = assay.plot_missing_values(
    run_id=last_run.run_id,
    file_type="raw",
    title=f"Missing Values ? run {last_run.run_number}",
)
bokeh_show(fig)
